# 과제 1. Data Processing with Pandas

2026-2 인공지능의 이해

Santander Product Recommendation 경진대회 데이터 중 `train_ver2.csv`만 사용한다.
데이터는 2015-01-28부터 2016-05-28까지의 월별 고객 스냅샷이다.
상품 보유 여부는 25번째부터 48번째 열, 이름이 `ind_*_ult1`인 컬럼이며 값은 0 또는 1이다.


## 데이터 불러오기

분석에 필요한 열만 읽고, 나이·소득·범주형 문자열의 공백을 정리한다.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import chi2_contingency

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 80)


def find_train_csv() -> Path:
    candidates = [
        Path("../data/train_ver2.csv"),
        Path("data/train_ver2.csv"),
        Path("/Users/apple/Desktop/oss6/data/train_ver2.csv"),
        Path.home() / "Downloads" / "train_ver2.csv",
        Path.home() / "Desktop" / "train_ver2.csv",
    ]
    for path in candidates:
        if path.is_file():
            return path.resolve()
    raise FileNotFoundError("train_ver2.csv 를 찾지 못했습니다.")


train_path = find_train_csv()
print("data:", train_path)

header_cols = list(pd.read_csv(train_path, nrows=0).columns)
product_cols = header_cols[24:48]
if len(product_cols) != 24 or not all(
    name.startswith("ind_") and name.endswith("_ult1") for name in product_cols
):
    raise ValueError(f"상품 컬럼 확인 실패: {product_cols}")

demo_cols = [
    "fecha_dato",
    "ncodpers",
    "ind_empleado",
    "sexo",
    "age",
    "nomprov",
    "renta",
    "tiprel_1mes",
    "segmento",
]
print("reading train_ver2.csv ...")
df = pd.read_csv(train_path, usecols=demo_cols + product_cols, low_memory=False)

df["fecha_dato"] = pd.to_datetime(df["fecha_dato"])
df["age"] = pd.to_numeric(df["age"].astype(str).str.strip(), errors="coerce")
df["renta"] = pd.to_numeric(df["renta"], errors="coerce")
for col in ["ind_empleado", "sexo", "nomprov", "tiprel_1mes", "segmento"]:
    df[col] = df[col].astype("string").str.strip()
    df.loc[df[col].isin(["", "NA", "nan", "None"]), col] = pd.NA
for col in product_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("shape:", df.shape)
print("period:", df["fecha_dato"].min().date(), "~", df["fecha_dato"].max().date())
print("months:", int(df["fecha_dato"].nunique()), "customers:", int(df["ncodpers"].nunique()))
print("product columns:")
print(product_cols)


data: /Users/apple/Desktop/oss6/data/train_ver2.csv
reading train_ver2.csv ...


shape: (13647309, 33)
period: 2015-01-28 ~ 2016-05-28


months: 17 customers: 956645
product columns:
['ind_ahor_fin_ult1', 'ind_aval_fin_ult1', 'ind_cco_fin_ult1', 'ind_cder_fin_ult1', 'ind_cno_fin_ult1', 'ind_ctju_fin_ult1', 'ind_ctma_fin_ult1', 'ind_ctop_fin_ult1', 'ind_ctpp_fin_ult1', 'ind_deco_fin_ult1', 'ind_deme_fin_ult1', 'ind_dela_fin_ult1', 'ind_ecue_fin_ult1', 'ind_fond_fin_ult1', 'ind_hip_fin_ult1', 'ind_plan_fin_ult1', 'ind_pres_fin_ult1', 'ind_reca_fin_ult1', 'ind_tjcr_fin_ult1', 'ind_valo_fin_ult1', 'ind_viv_fin_ult1', 'ind_nomina_ult1', 'ind_nom_pens_ult1', 'ind_recibo_ult1']


## Q1. 마지막 달 거주 도시별 통계

2016-05-28 스냅샷에서 `nomprov`별로 다음을 계산한다.

- 고객 수
- 평균 소득 (`renta`)
- 소득 중앙값
- 평균 나이 (`age`)

고객 수가 50명 이상인 도시만 남기고, 평균 소득이 높은 상위 5개 도시를 출력한다.
소득 평균과 중앙값은 결측 소득을 제외하고 계산한다.


In [2]:
last_date = pd.Timestamp("2016-05-28")
last = df.loc[df["fecha_dato"].eq(last_date)].copy()
print("마지막 달 행 수:", len(last))
print("마지막 달 고객 수:", int(last["ncodpers"].nunique()))

province = last.groupby("nomprov", dropna=True)
city_stats = pd.DataFrame(
    {
        "고객 수": province["ncodpers"].nunique(),
        "평균 소득": province["renta"].mean(),
        "소득 중앙값": province["renta"].median(),
        "평균 나이": province["age"].mean(),
    }
)
city_stats = city_stats.loc[city_stats["고객 수"] >= 50]
city_stats = city_stats.sort_values(["평균 소득", "고객 수"], ascending=[False, False])
top5_cities = city_stats.head(5).reset_index().rename(columns={"nomprov": "거주 도시"})

print("고객 수 50명 이상인 도시 수:", len(city_stats))
with pd.option_context("display.float_format", "{:,.2f}".format):
    display(top5_cities)


마지막 달 행 수: 931453
마지막 달 고객 수: 931453


고객 수 50명 이상인 도시 수: 52


,거주 도시,고객 수,평균 소득,소득 중앙값,평균 나이
0,CEUTA,513,"200,515.89","128,699.58",44.01
1,MADRID,298841,"178,362.99","138,024.14",45.21
2,"BALEARS, ILLES",9143,"171,890.95","122,837.22",39.75
3,BARCELONA,88754,"164,584.84","130,877.16",39.87
4,MELILLA,641,"149,071.94","117,718.37",45.33


## Q2. 전월 대비 신규 가입 상품

고객마다 바로 이전 기록과 비교해 상품이 0에서 1로 바뀐 경우를 신규 가입으로 센다.
이전 기록이 달력상 한 달 전일 때만 집계한다. 그래서 해당 고객의 첫 달과, 한 달 이상 비어 있는 간격은 신규 가입에 넣지 않는다.
월별·상품별 건수를 구한 다음, 전체 기간 합계가 가장 큰 상품 5개의 컬럼 이름을 출력한다.

이전 달 값은 `groupby.shift`로 구한다. `.agg`와 `.diff`는 사용하지 않는다.


In [3]:
history = df.loc[:, ["fecha_dato", "ncodpers", *product_cols]].copy()
history = history.sort_values(["ncodpers", "fecha_dato"], kind="mergesort")

dup_count = int(history.duplicated(["ncodpers", "fecha_dato"]).sum())
print("중복된 고객-월 행 수:", dup_count)
if dup_count:
    history = (
        history.groupby(["ncodpers", "fecha_dato"], sort=False)[product_cols]
        .max()
        .reset_index()
        .sort_values(["ncodpers", "fecha_dato"], kind="mergesort")
    )

history["month_ord"] = history["fecha_dato"].dt.year * 12 + history["fecha_dato"].dt.month
grouped = history.groupby("ncodpers", sort=False)
prev_products = grouped[product_cols].shift(1)
prev_month = grouped["month_ord"].shift(1)
consecutive = history["month_ord"].eq(prev_month + 1)

is_new = history[product_cols].eq(1) & prev_products.eq(0)
is_new = is_new.astype("int8").mul(consecutive.astype("int8"), axis=0)

monthly_new = is_new.groupby(history["fecha_dato"], sort=True).sum()
monthly_new.index = monthly_new.index.strftime("%Y-%m-%d")
monthly_new = monthly_new.astype("int64")
total_new = monthly_new.sum(axis=0).sort_values(ascending=False)
top5_products = total_new.head(5).to_frame("신규 가입 건수")

print("월별·상품별 신규 가입 건수")
display(monthly_new)
print("전체 기간 신규 가입 상위 5개 상품")
display(top5_products)


중복된 고객-월 행 수: 0


월별·상품별 신규 가입 건수


,ind_ahor_fin_ult1,ind_aval_fin_ult1,ind_cco_fin_ult1,ind_cder_fin_ult1,ind_cno_fin_ult1,ind_ctju_fin_ult1,ind_ctma_fin_ult1,ind_ctop_fin_ult1,ind_ctpp_fin_ult1,ind_deco_fin_ult1,ind_deme_fin_ult1,ind_dela_fin_ult1,ind_ecue_fin_ult1,ind_fond_fin_ult1,ind_hip_fin_ult1,ind_plan_fin_ult1,ind_pres_fin_ult1,ind_reca_fin_ult1,ind_tjcr_fin_ult1,ind_valo_fin_ult1,ind_viv_fin_ult1,ind_nomina_ult1,ind_nom_pens_ult1,ind_recibo_ult1
fecha_dato,,,,,,,,,,,,,,,,,,,,,,,,
2015-01-28,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2015-02-28,1,1,2132,12,1884,7,291,249,171,348,33,724,1472,451,5,27,9,217,3471,259,8,5659,5656,8285
2015-03-28,0,1,2293,15,1786,18,272,284,165,481,23,814,1594,570,7,24,12,217,5076,205,4,4479,4473,9135
2015-04-28,0,1,2366,12,1984,10,217,278,167,263,26,775,1459,492,4,22,17,871,4627,194,5,2663,5628,8346
2015-05-28,0,0,2264,7,1900,10,205,289,172,238,25,885,1461,396,9,21,11,343,4071,201,4,3369,3345,7085
2015-06-28,0,0,6588,9,1857,7,210,213,151,289,33,939,1204,245,4,19,8,2931,4655,152,3,5070,8135,9023
2015-07-28,0,0,3056,10,2771,18,191,252,164,265,24,1138,1085,204,6,27,5,589,4886,232,9,4408,4850,9172
2015-08-28,0,0,5730,10,2412,37,240,231,145,322,33,1024,924,223,1,22,13,264,4379,445,2,2950,2968,7115
2015-09-28,0,0,4389,7,1775,22,471,191,122,387,30,1348,1072,130,8,21,13,271,4463,328,2,4683,4753,11210


전체 기간 신규 가입 상위 5개 상품


,신규 가입 건수
ind_recibo_ult1,153146
ind_nom_pens_ult1,84709
ind_nomina_ult1,73753
ind_tjcr_fin_ult1,69118
ind_cco_fin_ult1,69103


## Q3. 보유 상품 3개 이상 여부와 가장 관련 있는 기준

마지막 달(2016-05-28)에 고객이 보유한 상품 수를 세고, 3개 이상인지 3개 미만인지로 나눈다.
상품 컬럼의 결측은 보유하지 않은 것(0)으로 두고 합산한다.

아래 다섯 기준과 이 이분 변수에 대해 카이제곱 검정을 한다.

1. 성별 (`sexo`: H / V)
2. 나이 구간: 65세 이상 senior, 30세 이상 65세 미만 adult, 30세 미만 young
3. 세그먼트 (`segmento`)
4. 직원 구분 (`ind_empleado`: A / B / F / N / P)
5. 월초 고객 관계 (`tiprel_1mes`: A / I / P / R)

각 검정에서는 해당 기준이 비어 있는 고객을 제외한다.
2×2 표에만 적용되는 Yates 보정은 끄고(`correction=False`) 다섯 기준을 같은 방식으로 계산한다.

원시 카이제곱 통계량은 범주 수와 표본 수가 다르면 연관 강도를 직접 비교할 수 없다.
그래서 카이제곱 통계량으로 Cramér's V를 계산하고, V가 가장 큰 기준을 관련성이 가장 큰 기준으로 선택한다.

\(V = \sqrt{\chi^2 / (n \cdot (\min(r, c) - 1))}\)


In [4]:
last = df.loc[df["fecha_dato"].eq(pd.Timestamp("2016-05-28"))].copy()
last["n_products"] = last[product_cols].fillna(0).sum(axis=1)
last["many_products"] = np.where(last["n_products"] >= 3, "3개 이상", "3개 미만")
print(last["many_products"].value_counts())
print("평균 보유 상품 수: {:.4f}".format(last["n_products"].mean()))

age_group = pd.cut(
    last["age"],
    bins=[-np.inf, 30, 65, np.inf],
    labels=["young", "adult", "senior"],
    right=False,
)


def keep(series, allowed):
    return series.where(series.isin(allowed))


features = [
    ("성별 (sexo)", keep(last["sexo"], ["H", "V"])),
    ("나이 구간", age_group),
    ("세그먼트 (segmento)", last["segmento"]),
    ("직원 구분 (ind_empleado)", keep(last["ind_empleado"], list("ABFNP"))),
    ("월초 관계 (tiprel_1mes)", keep(last["tiprel_1mes"], list("AIPR"))),
]

rows = []
for name, feature in features:
    valid = feature.notna()
    table = pd.crosstab(feature.loc[valid], last.loc[valid, "many_products"])
    for column in ["3개 미만", "3개 이상"]:
        if column not in table.columns:
            table[column] = 0
    table = table[["3개 미만", "3개 이상"]]
    chi2, pvalue, dof, _expected = chi2_contingency(table.to_numpy(), correction=False)
    sample_size = int(table.to_numpy().sum())
    n_rows, n_cols = table.shape
    cramer_v = float(np.sqrt(chi2 / (sample_size * (min(n_rows, n_cols) - 1))))
    rows.append(
        {
            "기준": name,
            "chi2": chi2,
            "자유도": int(dof),
            "p-value": pvalue,
            "Cramers V": cramer_v,
            "표본 수": sample_size,
        }
    )
    print("\n[{}]".format(name))
    display(table)

association = pd.DataFrame(rows).sort_values("Cramers V", ascending=False).reset_index(drop=True)
print("\n카이제곱 검정 비교")
association_view = association.copy()
association_view["chi2"] = association_view["chi2"].map(lambda value: f"{value:,.2f}")
association_view["p-value"] = association_view["p-value"].map(lambda value: f"{value:.3e}")
association_view["Cramers V"] = association_view["Cramers V"].map(lambda value: f"{value:.6f}")
display(association_view)

best = association.iloc[0]
print(
    "가장 관련성이 큰 기준: {} (Cramér's V = {:.6f}, chi2 = {:.2f}, p = {:.3e})".format(
        best["기준"], best["Cramers V"], best["chi2"], best["p-value"]
    )
)


many_products
3개 미만    808432
3개 이상    123021
Name: count, dtype: int64
평균 보유 상품 수: 1.3318



[성별 (sexo)]


many_products,3개 미만,3개 이상
sexo,,
H,379129,46737
V,429298,76284



[나이 구간]


many_products,3개 미만,3개 이상
age,,
young,340597,8013
adult,388734,102166
senior,79101,12842


...


[월초 관계 (tiprel_1mes)]


many_products,3개 미만,3개 이상
tiprel_1mes,,
A,271352,121255
I,532690,1739
P,330,0
R,76,1



카이제곱 검정 비교


,기준,chi2,자유도,p-value,Cramers V,표본 수
0,월초 관계 (tiprel_1mes),"183,808.59",3,0.000e+00,0.445184,927443
1,세그먼트 (segmento),"124,374.55",2,0.000e+00,0.366800,924428
2,나이 구간,"61,002.20",2,0.000e+00,0.255913,931453
3,성별 (sexo),"3,412.59",1,0.000e+00,0.060529,931448
4,직원 구분 (ind_empleado),"1,337.13",3,1.295e-289,0.037888,931452


가장 관련성이 큰 기준: 월초 관계 (tiprel_1mes) (Cramér's V = 0.445184, chi2 = 183808.59, p = 0.000e+00)
